# 07 — Per-mode entropic schedule (theoretical bridge to ESDM)

This notebook demonstrates the per-mode entropic noise schedule from the paper §5:

$$\tau_k(t) = \nu_k \cdot t, \qquad \bar\alpha_k(\tau_k) = \cos^2\!\left(\frac{\pi}{2}\cdot\frac{\tau_k}{\tau_{k,\max}}\right)$$

where $\tau_{k,\max} = \nu_k \cdot T$ and $T$ is the reference horizon.

**Key mathematical insight:** $\nu_k$ cancels in the ratio $\tau_k/\tau_{k,\max} = t/T$, so all modes have identical $\bar\alpha_k$ in external time. The per-mode differentiation comes through the **heat-death metric** $\sum_k \nu_k \cdot \mathrm{mmse}_k(t)$, which is weighted by $\nu_k$.

This is the theoretical bridge from ALD-SC (minimal spectral-chart model) back to ESDM (full entropic clock). It is a thin reproducible stub — the full wave recurrence, vibrational pump, and density matrices remain Phase 4 / future work.

In [ ]:
import sys
sys.path.insert(0, "../src")

import torch
import matplotlib.pyplot as plt
from ald_sc.build_prior import build_arrow_prior
from ald_sc.spectral_schedule import SpectralSchedule

torch.manual_seed(3407)

## 1. Build the spectral schedule from a frozen prior

In [ ]:
F, q = 32, 8
embeddings = torch.randn(64, F)
prior = build_arrow_prior(embeddings, q=q, k=4)
sched = SpectralSchedule(prior, horizon=1.0)

print(f"q = {sched.q}")
print(f"ν (eigenvalues): {sched.nu.tolist()}")
print(f"τ_k_max = {sched.tau_k_max.tolist()}")
print(f"Trainable parameters: {len(list(sched.parameters()))}")

## 2. Per-mode entropic time $\tau_k(t) = \nu_k \cdot t$

High-$\nu_k$ modes accumulate entropic time faster.

In [ ]:
ts = torch.linspace(0, 1, 100)
tau_ks = torch.stack([sched.tau_k(t) for t in ts])

fig, ax = plt.subplots(figsize=(8, 4))
for k in range(q):
    ax.plot(ts.numpy(), tau_ks[:, k].numpy(), alpha=0.7, label=f"k={k} (ν={sched.nu[k]:.3f})")
ax.set_xlabel("External time t")
ax.set_ylabel(r"$\tau_k(t) = \nu_k \cdot t$")
ax.set_title("Per-mode entropic time")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig("../results/07_tau_k.png", dpi=150)
plt.show()

## 3. Per-mode $\bar\alpha_k$ — all modes identical in external time

Because $\nu_k$ cancels in $\tau_k / \tau_{k,\max} = t/T$, all modes have the same $\bar\alpha_k$ at any given external time $t$.

In [ ]:
ab_ks = torch.stack([sched.alpha_bar_k(t) for t in ts])

fig, ax = plt.subplots(figsize=(8, 4))
for k in range(q):
    ax.plot(ts.numpy(), ab_ks[:, k].numpy(), alpha=0.5)
ax.plot(ts.numpy(), ab_ks[:, 0].numpy(), "k--", linewidth=2, label="All modes (identical)")
ax.set_xlabel("External time t")
ax.set_ylabel(r"$\bar\alpha_k(t)$")
ax.set_title(r"Per-mode $\bar\alpha_k$ — identical in external time ($\nu_k$ cancels)")
ax.legend()
plt.tight_layout()
plt.savefig("../results/07_alpha_bar_k.png", dpi=150)
plt.show()

## 4. Entropy rate $dS_k/dt = -\nu_k$

Each Laplacian eigenvalue *is* the entropy exchange rate for its mode.

In [ ]:
rates = sched.entropy_rate()
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(q), rates.numpy())
ax.set_xlabel("Mode k")
ax.set_ylabel(r"$dS_k/dt = -\nu_k$")
ax.set_title("Entropy exchange rate per mode")
plt.tight_layout()
plt.savefig("../results/07_entropy_rate.png", dpi=150)
plt.show()

## 5. Heat-death metric and stopping criterion

$$\mathrm{heat\ death} \iff \sum_{k=1}^{q} \nu_k \cdot \mathrm{mmse}_k(t) < \varepsilon$$

The metric is weighted by $\nu_k$, so high-$\nu_k$ modes dominate. The process terminates when no further information is being resolved.

In [ ]:
metrics = [sched.heat_death_metric(t).item() for t in ts]

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(ts.numpy(), metrics, linewidth=2)
ax.axhline(y=sched.eps, color="r", linestyle="--", label=f"ε = {sched.eps}")
ax.set_xlabel("External time t")
ax.set_ylabel(r"$\sum_k \nu_k \cdot \bar\alpha_k(t)$")
ax.set_title("Heat-death metric")
ax.legend()
plt.tight_layout()
plt.savefig("../results/07_heat_death.png", dpi=150)
plt.show()

# Find the heat-death time
for t in ts:
    if sched.is_heat_death(t):
        print(f"Heat death at t = {t:.4f}")
        break
else:
    print("No heat death in [0, 1]")